# 01 — WavLM embedding extraction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SepidehNorouziMotlagh/speaker-disentanglement-grl/blob/main/notebooks/01_extract_wavlm_embeddings.ipynb)

Extracts frozen `microsoft/wavlm-base-plus` embeddings for CREMA-D.
Produces one 768-d mean-pooled embedding per recording under
`embeddings/{train,validation,test}.npz`.

**Prerequisites**
- `AudioWAV/` populated with CREMA-D WAV files (sibling to `embeddings/`)
- `cremad_splits.csv` produced by `python -m src.data.split_dataset`

**Runtime** ~45 min on Colab T4. Resumable — re-run after a disconnect
and completed chunks will be skipped.

## 1. Setup

In [ ]:
!pip install -q transformers soundfile librosa pandas scikit-learn tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Edit `PROJECT_DIR` to your Drive location. `sys.path` is extended so
`from src.data.extract_embeddings import ...` resolves.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/content/drive/MyDrive/speaker-disentanglement-grl')
AUDIO_DIR = PROJECT_DIR / 'AudioWAV'
METADATA_FILE = PROJECT_DIR / 'cremad_splits.csv'
OUTPUT_DIR = PROJECT_DIR / 'embeddings'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))

print('Audio:', AUDIO_DIR)
print('Output:', OUTPUT_DIR)

## 3. Load frozen WavLM

In [ ]:
import torch
from transformers import AutoFeatureExtractor, WavLMModel

MODEL_NAME = 'microsoft/wavlm-base-plus'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
wavlm = WavLMModel.from_pretrained(MODEL_NAME).to(device).eval()
for p in wavlm.parameters():
    p.requires_grad = False

print('Device:', device)

## 4. Prepare metadata and label maps

In [ ]:
import json
import pandas as pd

df = pd.read_csv(METADATA_FILE, dtype={'speaker_id': str})
df['speaker_id'] = df['speaker_id'].str.strip()
df['filename'] = df['filename'].astype(str)

EMOTION_TO_ID = {'anger': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5}

train_speakers = sorted(df.loc[df['split'] == 'train', 'speaker_id'].unique(), key=int)
assert len(train_speakers) == 64, f'Expected 64 training speakers, got {len(train_speakers)}'
SPEAKER_TO_ID = {spk: i for i, spk in enumerate(train_speakers)}

assert df['filename'].is_unique
assert set(df['split'].unique()) == {'train', 'validation', 'test'}
assert set(df['emotion'].unique()).issubset(EMOTION_TO_ID)

missing = [n for n in df['filename'] if not (AUDIO_DIR / n).is_file()]
if missing:
    raise FileNotFoundError(f'{len(missing)} audio files missing. First: {missing[:5]}')

with open(OUTPUT_DIR / 'label_maps.json', 'w') as f:
    json.dump({'emotion_to_id': EMOTION_TO_ID, 'train_speaker_to_id': SPEAKER_TO_ID}, f, indent=2)

df = df.sort_values('filename').reset_index(drop=True)
print(f'{len(df)} recordings, {len(SPEAKER_TO_ID)} training speakers')

## 5. Extract embeddings

Resumable: each 64-file chunk is written atomically. Completed splits
are skipped on re-run.

In [ ]:
from src.data.extract_embeddings import extract_split

for split in ['train', 'validation', 'test']:
    extract_split(
        split_name=split, df=df,
        audio_dir=AUDIO_DIR, output_dir=OUTPUT_DIR,
        wavlm=wavlm, feature_extractor=feature_extractor, device=device,
        emotion_to_id=EMOTION_TO_ID, speaker_to_id=SPEAKER_TO_ID,
    )

## 6. Verify output

In [ ]:
import numpy as np

for split in ['train', 'validation', 'test']:
    with np.load(OUTPUT_DIR / f'{split}.npz', allow_pickle=False) as data:
        print(
            f'{split:10s} | embeddings {data["embeddings"].shape} | '
            f'speakers {len(np.unique(data["speaker_ids"]))}'
        )

In [ ]:
emotion_counts = pd.crosstab(df['split'], df['emotion'])
print(emotion_counts)
print('\nProportions:')
print(emotion_counts.div(emotion_counts.sum(axis=1), axis=0).round(3))